# 🐙 Cipher — Code Kraken Training Notebook

Train the Cipher local AI model on a **free T4 GPU** using Unsloth.

**Pipeline:** SFT → SimPO → GRPO → Export GGUF

**Base model:** Gemma 4 E4B (4.5B params, fits in 15GB T4 VRAM)

**Time:** ~2-4 hours total on free Colab T4

---

## Setup

In [ ]:
# Install Unsloth (Colab-optimized)
!pip install unsloth[colab-new] -q
!pip install trl datasets -q

In [ ]:
# Verify GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
print(f'Torch: {torch.__version__}')

## Stage 1: Load Gemma 4 E4B with Unsloth QLoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/gemma-4-E4B-it-bnb-4bit',
    max_seq_length=4096,
    load_in_4bit=True,
    dtype=None,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

print('🐙 Model loaded! LoRA adapters attached.')

## Stage 2: Generate Training Data (Zero-Cost Templates)

In [ ]:
# Generate template-based training data
# For production: use frontier model distillation (generate-sft.py --mode distill)
!python training/data-synthesis/generate-sft.py --mode templates --count 1000 --output /tmp/cipher-data/

In [ ]:
# Load training data
import json
from datasets import Dataset

examples = []
for f in ['/tmp/cipher-data/frontend-tutorials.jsonl',
          '/tmp/cipher-data/design-critiques.jsonl',
          '/tmp/cipher-data/persona-conversations.jsonl',
          '/tmp/cipher-data/tool-calling-examples.jsonl']:
    try:
        with open(f) as fh:
            for line in fh:
                examples.append(json.loads(line))
    except: pass

# Format for Gemma 4 ChatML
def format_example(ex):
    text = ''
    for msg in ex.get('messages', []):
        role = msg['role']
        content = msg['content']
        if role == 'system':
            text += f'<start_of_turn>system\n{content}<end_of_turn>\n'
        elif role == 'user':
            text += f'<start_of_turn>user\n{content}<end_of_turn>\n'
        elif role == 'assistant':
            text += f'<start_of_turn>model\n{content}<end_of_turn>\n'
    return {'text': text}

dataset = Dataset.from_list([format_example(ex) for ex in examples])
print(f'🐙 Training dataset: {len(dataset)} examples')

## Stage 3: SFT Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=TrainingArguments(
        output_dir='/tmp/cipher-sft',
        num_train_epochs=2,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        weight_decay=0.01,
        bf16=True,
        optim='adamw_8bit',
        logging_steps=25,
        save_steps=100,
        seed=42,
        report_to='none',
    ),
    dataset_text_field='text',
    max_seq_length=4096,
    packing=True,
)

print('🐙 Starting SFT training...')
trainer.train()
print('🐙 SFT training complete!')

## Stage 4: Export GGUF for Ollama

In [ ]:
# Export to GGUF (Q4_K_M for 6GB VRAM)
print('🐙 Exporting GGUF...')
model.save_pretrained_gguf(
    '/tmp/cipher-gguf',
    tokenizer,
    quantization_method=['q4_k_m', 'q5_k_m'],
)
print('🐙 GGUF exported!')

# Show file sizes
import os
for f in os.listdir('/tmp/cipher-gguf'):
    size = os.path.getsize(f'/tmp/cipher-gguf/{f}') / 1e9
    print(f'  {f}: {size:.2f} GB')

In [ ]:
# Optional: Push to HuggingFace Hub
# model.push_to_hub_gguf(
#     'kr8tiv/kin-cipher-GGUF',
#     tokenizer,
#     quantization_method=['q4_k_m', 'q5_k_m'],
#     token='hf_...',
# )

print('🐙 Done! Download the GGUF from /tmp/cipher-gguf/')
print('Then on your machine:')
print('  ollama create kin-cipher -f Modelfile')
print('  ollama run kin-cipher')